In [1]:
!pip install kagglehub

In [2]:
import kagglehub

path = kagglehub.dataset_download("ajverse/customer-support-tickets-crm-dataset")

print(path)

100%|██████████| 2.08M/2.08M [00:01<00:00, 1.42MB/s]

Extracting files...
C:\Users\user\.cache\kagglehub\datasets\ajverse\customer-support-tickets-crm-dataset\versions\2


In [22]:
import os
import shutil
from pathlib import Path

download_path = Path(path)

target_dir = Path("../data/raw")
target_dir.mkdir(parents=True, exist_ok=True)


In [23]:
csv_files = list(download_path.glob("*.csv"))

for csv in csv_files:
    shutil.copy(csv, target_dir / csv.name)
    print("Copied:", csv.name)

Copied: customer_support_tickets.csv
Copied: enhanced_customer_support_data.csv


In [24]:
import pandas as pd

file1 = "../data/raw/customer_support_tickets.csv"
file2 = "../data/raw/enhanced_customer_support_data.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

print(df1.shape)
print(df2.shape)

(20000, 12)
(20000, 12)


In [25]:
df1.head()

,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,Elena Rodriguez,5
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,Anya Sharma,5
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,David Kim,5


In [26]:
df2.head()

,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,Elena Rodriguez,5
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,Anya Sharma,5
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,David Kim,5


In [27]:
print( df1.equals(df2))

True


In [28]:
df=df1

In [29]:
df["Priority_Level"].value_counts()

Priority_Level
Low         7716
Medium      7570
High        3416
Critical    1298
Name: count, dtype: int64

In [30]:
df.isnull().sum()

Ticket_ID                0
Customer_Name            0
Customer_Email           0
Ticket_Subject           0
Ticket_Description       0
Issue_Category           0
Priority_Level           0
Ticket_Channel           0
Submission_Date          0
Resolution_Time_Hours    0
Assigned_Agent           0
Satisfaction_Score       0
dtype: int64

In [31]:
print(df["Ticket_ID"].duplicated().sum())

0


In [32]:
df["ticket_text"] = (
    "Subject: " + df["Ticket_Subject"].astype(str) +
    " Description: " + df["Ticket_Description"].astype(str)
)

df[["Ticket_Subject", "Ticket_Description", "ticket_text"]].head()

,Ticket_Subject,Ticket_Description,ticket_text
0,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",Subject: Hours of operation - Individual Descr...
1,Data not syncing - Card,"Hi Support, The application crashes every time...",Subject: Data not syncing - Card Description: ...
2,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Subject: 2FA issues - Question Description: Hi...
3,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Subject: Login failed - Let Description: Hi Su...
4,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Subject: Refund status - Attention Description...


In [34]:
priority_map = {
    "Low": 0,
    "Medium": 1,
    "High": 2,
    "Critical": 3
}

df["assigned_priority_score"] = df["Priority_Level"].map(priority_map)

df[["Priority_Level", "assigned_priority_score"]].sample(6)

,Priority_Level,assigned_priority_score
16999,Low,0
12769,High,2
17913,Low,0
8106,High,2
18543,Low,0
6502,Low,0


In [35]:
df.groupby("Priority_Level")["Resolution_Time_Hours"].describe()

,count,mean,std,min,25%,50%,75%,max
Priority_Level,,,,,,,,
Critical,1298.0,12.068567,11.383106,1.0,4.0,9.0,16.0,80.0
High,3416.0,24.520492,23.244518,1.0,7.0,17.0,35.0,120.0
Low,7716.0,45.168351,36.649743,1.0,15.0,34.0,68.0,120.0
Medium,7570.0,44.472919,36.812959,1.0,14.0,33.0,67.0,120.0


In [37]:
print(df["Issue_Category"].value_counts())

print(df["Ticket_Channel"].value_counts())

Issue_Category
Technical          5918
Billing            5036
Account            4081
General Inquiry    3925
Fraud              1040
Name: count, dtype: int64
Ticket_Channel
Chat        6693
Email       6656
Web Form    6651
Name: count, dtype: int64


In [38]:
df[["assigned_priority_score", "Resolution_Time_Hours"]].corr()

,assigned_priority_score,Resolution_Time_Hours
assigned_priority_score,1.000000,-0.262923
Resolution_Time_Hours,-0.262923,1.000000


In [40]:
df["Resolution_Time_Hours"].corr(df["Satisfaction_Score"])

-0.002962150495083069

In [41]:
df.select_dtypes(include="number").corr()

,Resolution_Time_Hours,Satisfaction_Score,assigned_priority_score
Resolution_Time_Hours,1.000000,-0.002962,-0.262923
Satisfaction_Score,-0.002962,1.000000,-0.005262
assigned_priority_score,-0.262923,-0.005262,1.000000


In [43]:
df.groupby("Assigned_Agent")["Satisfaction_Score"].agg(
    ["count", "mean", "median", "min", "max"]
).sort_values(by="mean", ascending=False)

,count,mean,median,min,max
Assigned_Agent,,,,,
Chloe Adams,4006,3.737644,4.0,1,5
Ben Carter,4010,3.727431,4.0,1,5
Elena Rodriguez,3970,3.720655,4.0,1,5
David Kim,3989,3.718225,4.0,1,5
Anya Sharma,4025,3.714534,4.0,1,5


In [44]:
df["Assigned_Agent"].value_counts()

Assigned_Agent
Anya Sharma        4025
Ben Carter         4010
Chloe Adams        4006
David Kim          3989
Elena Rodriguez    3970
Name: count, dtype: int64

In [45]:
pd.crosstab(
    df["Assigned_Agent"],
    df["Priority_Level"]
)

Priority_Level,Critical,High,Low,Medium
Assigned_Agent,,,,
Anya Sharma,248,679,1585,1513
Ben Carter,293,676,1557,1484
Chloe Adams,241,678,1534,1553
David Kim,278,677,1510,1524
Elena Rodriguez,238,706,1530,1496


In [46]:
df[["Ticket_Subject", "Ticket_Description", "Issue_Category", "Priority_Level"]].sample(10)

,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level
9186,App crashing - Determine,"Hi Support, We are receiving a 500 Internal Se...",Technical,High
18867,Suspicious charge - Water,"Hi Support, My subscription renewed automatica...",Billing,Low
2121,Change email - Must,"Hi Support, My profile picture is not updating...",Account,Low
9704,Login failed - Enjoy,"Hi Support, How do I install the latest patch ...",Technical,Low
926,2FA issues - Education,"Hi Support, My profile picture is not updating...",Account,Medium
9981,Hours of operation - Learn,"Hi Support, I would like to request a demo for...",General Inquiry,Low
9234,Pricing tiers - Born,"Hi Support, Is there a roadmap for new feature...",General Inquiry,Medium
2064,Login failed - Goal,"Hi Support, We are receiving a 500 Internal Se...",Technical,Medium
5509,Payment failed - Some,"Hi Support, Can you send me the invoice for tr...",Billing,High
8581,Change email - Without,"Hi Support, Please help me change the email ad...",Account,Low


In [49]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 3),
    min_df=20
)

X = vectorizer.fit_transform(df["ticket_text"])

phrase_counts = pd.DataFrame({
    "phrase": vectorizer.get_feature_names_out(),
    "count": X.toarray().sum(axis=0)
})

phrase_counts.sort_values("count", ascending=False).head(50)

,phrase,count
1895,support,20939
1814,subject,20155
499,description hi support,20000
498,description hi,20000
828,hi support,20000
497,description,20000
827,hi,20000
450,data,3692
21,account,3622
1296,password,2338


In [51]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

custom_stop_words = [
    "subject", "description", "hi", "support", "team", "just",
    "receiving", "open", "time"
]

vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(2, 4),
    min_df=10
)

X = vectorizer.fit_transform(df["ticket_text"])

phrase_counts = pd.DataFrame({
    "phrase": vectorizer.get_feature_names_out(),
    "count": X.toarray().sum(axis=0)
})
pattern = "|".join(custom_stop_words)

clean_phrases = phrase_counts[
    ~phrase_counts["phrase"].str.contains(pattern, case=False, regex=True)
]

clean_phrases.sort_values("count", ascending=False).head(60)

,phrase,count
368,change email,1386
1715,password reset,1385
586,delete account,1340
2890,windows 11,1053
1209,install latest,1053
1210,install latest patch,1053
1298,latest patch,1053
1299,latest patch windows,1053
1300,latest patch windows 11,1053
1211,install latest patch windows,1053


In [53]:
severity_keywords = {
    "critical": {
        "weight": 4,
        "phrases": [
            "api error 500",
            "500 internal server error",
            "internal server error",
            "server error api endpoint",
            "server error",
            "api endpoint"
        ]
    },

    "high": {
        "weight": 3,
        "phrases": [
            "application crashes",
            "screen freezes",
            "login failed",
            "dashboard loading",
            "dashboard loading data",
            "spinning wheel",
            "data hasn synced",
            "data hasn synced cloud",
            "hasn synced cloud 24 hours",
            "data syncing",
            "error api"
        ]
    },

    "medium": {
        "weight": 2,
        "phrases": [
            "password reset",
            "resetting password",
            "account resetting password",
            "installation issue",
            "change email",
            "delete account",
            "settings tab",
            "invoice transaction",
            "send invoice"
        ]
    },

    "low": {
        "weight": 1,
        "phrases": [
            "install latest",
            "install latest patch",
            "latest patch",
            "latest patch windows 11",
            "windows 11"
        ]
    }
}

In [54]:
import re 
def clean_text_for_matching(text):
    text = str(text).lower()
    text = text.replace("n't", " not")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [55]:
def rule_based_severity(text):
    clean_text = clean_text_for_matching(text)
    
    total_score = 0
    matched_evidence = []
    
    for severity_group, config in severity_keywords.items():
        weight = config["weight"]
        
        for phrase in config["phrases"]:
            clean_phrase = clean_text_for_matching(phrase)
            
            pattern = r"\b" + re.escape(clean_phrase) + r"\b"
            
            if re.search(pattern, clean_text):
                total_score += weight
                
                matched_evidence.append({
                    "signal": "keyword",
                    "value": phrase,
                    "weight": weight,
                    "group": severity_group
                })
    
    if total_score >= 8:
        severity_score = 3
    elif total_score >= 5:
        severity_score = 2
    elif total_score >= 2:
        severity_score = 1
    else:
        severity_score = 0
    
    return total_score, severity_score, matched_evidence

In [56]:
rule_results = df["ticket_text"].apply(rule_based_severity)

df["rule_score"] = rule_results.apply(lambda x: x[0])
df["rule_severity_score"] = rule_results.apply(lambda x: x[1])
df["rule_evidence"] = rule_results.apply(lambda x: x[2])

In [66]:
df["rule_score"].value_counts()

rule_score
0    10834
2     2487
4     2234
3     1725
7      832
6      822
5      723
1      343
Name: count, dtype: int64

In [67]:
df["rule_severity_score"].value_counts()

rule_severity_score
0    11177
3     3609
1     2746
2     2468
Name: count, dtype: int64

In [60]:
df["rule_evidence"].value_counts()

rule_evidence
[]                                                                                                                                                                                                                                                                                                                                                                                                                                                      12243
[{'signal': 'keyword', 'value': 'password reset', 'weight': 2, 'group': 'medium'}]                                                                                                                                                                                                                                                                                                                                                                       1032
[{'signal': 'keyword', 'value': 'change email', 'weight': 2, 'group': 'medium'}]              

In [61]:
severity_rules = [
    {
        "rule_name": "api_server_error",
        "severity_score": 3,
        "severity_label": "critical",
        "weight": 4,
        "phrases": [
            "api error 500",
            "500 internal server error",
            "internal server error",
            "server error api endpoint"
        ]
    },
    {
        "rule_name": "suspicious_activity",
        "severity_score": 3,
        "severity_label": "critical",
        "weight": 4,
        "phrases": [
            "suspicious activity",
            "suspicious"
        ]
    },
    {
        "rule_name": "application_crash_or_freeze",
        "severity_score": 2,
        "severity_label": "high",
        "weight": 3,
        "phrases": [
            "application crashes",
            "screen freezes",
            "spinning wheel"
        ]
    },
    {
        "rule_name": "login_failure",
        "severity_score": 2,
        "severity_label": "high",
        "weight": 3,
        "phrases": [
            "login failed"
        ]
    },
    {
        "rule_name": "dashboard_loading_issue",
        "severity_score": 2,
        "severity_label": "high",
        "weight": 3,
        "phrases": [
            "dashboard loading",
            "dashboard loading data",
            "loading data"
        ]
    },
    {
        "rule_name": "data_sync_issue",
        "severity_score": 2,
        "severity_label": "high",
        "weight": 3,
        "phrases": [
            "data has not synced",
            "data hasn't synced",
            "has not synced",
            "hasn't synced",
            "data syncing"
        ]
    },
    {
        "rule_name": "account_access_or_change",
        "severity_score": 1,
        "severity_label": "medium",
        "weight": 2,
        "phrases": [
            "password reset",
            "resetting password",
            "account resetting password",
            "change email"
        ]
    },
    {
        "rule_name": "account_closure",
        "severity_score": 1,
        "severity_label": "medium",
        "weight": 2,
        "phrases": [
            "delete account"
        ]
    },
    {
        "rule_name": "installation_issue",
        "severity_score": 1,
        "severity_label": "medium",
        "weight": 2,
        "phrases": [
            "installation issue"
        ]
    },
    {
        "rule_name": "billing_document_request",
        "severity_score": 1,
        "severity_label": "medium",
        "weight": 2,
        "phrases": [
            "invoice transaction",
            "send invoice"
        ]
    },
    {
        "rule_name": "patch_information",
        "severity_score": 0,
        "severity_label": "low",
        "weight": 1,
        "phrases": [
            "install latest patch",
            "latest patch",
            "latest patch windows 11",
            "windows 11"
        ]
    }
]

In [62]:
def rule_based_severity_v2(text):
    clean_text = clean_text_for_matching(text)
    
    total_score = 0
    matched_evidence = []
    matched_severity_scores = []
    
    for rule in severity_rules:
        for phrase in rule["phrases"]:
            clean_phrase = clean_text_for_matching(phrase)
            pattern = r"\b" + re.escape(clean_phrase) + r"\b"
            
            if re.search(pattern, clean_text):
                total_score += rule["weight"]
                matched_severity_scores.append(rule["severity_score"])
                
                matched_evidence.append({
                    "signal": "keyword",
                    "rule_name": rule["rule_name"],
                    "value": phrase,
                    "weight": rule["weight"],
                    "group": rule["severity_label"]
                })
                
                break  
    
    if len(matched_severity_scores) == 0:
        severity_score = 0
    else:
        severity_score = max(matched_severity_scores)
        
        # If medium/high signals cluster then we increase the severity
        if total_score >= 6 and severity_score < 3:
            severity_score += 1
    
    return total_score, severity_score, matched_evidence

In [63]:
rule_results = df["ticket_text"].apply(rule_based_severity_v2)

df["rule_score"] = rule_results.apply(lambda x: x[0])
df["rule_severity_score"] = rule_results.apply(lambda x: x[1])
df["rule_evidence"] = rule_results.apply(lambda x: x[2])

In [64]:
df["rule_severity_score"].value_counts().sort_index()

rule_severity_score
0    11177
1     2746
2     2468
3     3609
Name: count, dtype: int64

In [65]:
df["rule_score"].value_counts().sort_index()

rule_score
0    10834
1      343
2     2487
3     1725
4     2234
5      723
6      822
7      832
Name: count, dtype: int64

In [68]:
pd.crosstab(
    df["Priority_Level"],
    df["rule_severity_score"],
    normalize="index"
) * 100

rule_severity_score,0,1,2,3
Priority_Level,,,,
Critical,43.374422,2.234206,19.337442,35.053929
High,41.539813,9.865340,21.896956,26.697892
Low,65.603940,16.601866,6.194920,11.599274
Medium,54.597094,14.517834,13.091149,17.793923


In [69]:
df["Resolution_Time_Hours"].describe()

count    20000.000000
mean        39.230300
std         35.221884
min          1.000000
25%         11.000000
50%         27.000000
75%         58.000000
max        120.000000
Name: Resolution_Time_Hours, dtype: float64

In [72]:
q25 = df["Resolution_Time_Hours"].quantile(0.25)
q50 = df["Resolution_Time_Hours"].quantile(0.50)
q75 = df["Resolution_Time_Hours"].quantile(0.75)

print(q25, q50, q75)

11.0 27.0 58.0


In [73]:
def resolution_time_severity(hours):
    if hours <= q25:
        return 0
    elif hours <= q50:
        return 1
    elif hours <= q75:
        return 2
    else:
        return 3

In [74]:
df["resolution_time_score"] = df["Resolution_Time_Hours"].apply(resolution_time_severity)

In [76]:
df["resolution_time_score"].value_counts()

resolution_time_score
0    5144
2    5031
3    4938
1    4887
Name: count, dtype: int64

In [81]:
pd.crosstab(
    df["assigned_priority_score"],
    df["resolution_time_score"],
    normalize="index"
)*100

resolution_time_score,0,1,2,3
assigned_priority_score,,,,
0,19.803007,22.433904,27.436496,30.326594
1,20.541612,23.157199,26.301189,30.000000
2,37.236534,29.771663,23.653396,9.338407
3,60.785824,29.738059,8.859784,0.616333


In [77]:
pd.crosstab(
    df["rule_severity_score"],
    df["resolution_time_score"],
    normalize="index"
)*100

resolution_time_score,0,1,2,3
rule_severity_score,,,,
0,23.950971,24.022546,25.579315,26.447168
1,23.270211,23.889294,25.491624,27.348871
2,29.983793,24.756888,26.175041,19.084279
3,30.146855,25.907454,22.887226,21.058465


In [79]:
agreement = (df["rule_severity_score"] == df["resolution_time_score"]).mean()

print(agreement)

0.23695


In [80]:
relaxed_agreement = (
    abs(df["rule_severity_score"] - df["resolution_time_score"]) <= 1
).mean()

print( relaxed_agreement)

0.53355


In [82]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Ticket_ID                20000 non-null  object
 1   Customer_Name            20000 non-null  object
 2   Customer_Email           20000 non-null  object
 3   Ticket_Subject           20000 non-null  object
 4   Ticket_Description       20000 non-null  object
 5   Issue_Category           20000 non-null  object
 6   Priority_Level           20000 non-null  object
 7   Ticket_Channel           20000 non-null  object
 8   Submission_Date          20000 non-null  object
 9   Resolution_Time_Hours    20000 non-null  int64 
 10  Assigned_Agent           20000 non-null  object
 11  Satisfaction_Score       20000 non-null  int64 
 12  ticket_text              20000 non-null  object
 13  assigned_priority_score  20000 non-null  int64 
 14  rule_score               20000 non-nul

In [83]:
group_cols = ["Issue_Category", "Ticket_Channel"]

df["resolution_percentile"] = df.groupby(group_cols)["Resolution_Time_Hours"].rank(
    pct=True,
    method="average"
)

In [84]:
def resolution_signal_score(percentile):
    if percentile <= 0.25:
        return 0
    elif percentile <= 0.50:
        return 1
    elif percentile <= 0.75:
        return 2
    else:
        return 3

df["resolution_signal_score"] = df["resolution_percentile"].apply(resolution_signal_score)

In [85]:
df["resolution_signal_score"].value_counts()

resolution_signal_score
3    5010
2    5008
0    5008
1    4974
Name: count, dtype: int64

In [86]:
pd.crosstab(
    df["rule_severity_score"],
    df["resolution_signal_score"],
    normalize="index"
) * 100

resolution_signal_score,0,1,2,3
rule_severity_score,,,,
0,24.586204,24.845665,25.400376,25.167755
1,25.564457,24.763292,24.508376,25.163875
2,25.688817,24.392220,25.891410,24.027553
3,25.602660,25.353283,23.746190,25.297866


In [88]:
agreement = (
    df["rule_severity_score"] == df["resolution_signal_score"]
).mean()

relaxed_agreement = (
    abs(df["rule_severity_score"] - df["resolution_signal_score"]) <= 1
).mean()

print(agreement)
print(relaxed_agreement)

0.249
0.5592


In [89]:
!pip install sentence-transformers

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/588.9 kB ? eta -:--:--
   ----------------------------------- ---- 524.3/588.9 kB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 588.9/588.9 kB 1.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/693.4 kB ? eta -:--:--
   --------------- ------------------------ 262.1/693.4 kB ? eta -:--:--
   ---------------------------------------- 693.4/693.4 kB 2.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/123.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/123.0 MB 2.8 MB/s eta 0:00:44
    --------------------------------------- 1.8/123.0 MB 3.1 MB/s eta 0:00:39
    --------------------------------------- 2.6/123.0 MB 3.4 MB/s eta 0:00:36
   - -----------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.


In [90]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = df["ticket_text"].astype(str).tolist()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings.shape

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\user\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

(20000, 384)

In [91]:
np.save("../outputs/ticket_embeddings.npy", embeddings)

In [92]:
from sklearn.cluster import KMeans

n_clusters = 12

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=10
)

df["semantic_cluster"] = kmeans.fit_predict(embeddings)

df["semantic_cluster"].value_counts().sort_index()

semantic_cluster
0     1175
1     1245
2     1053
3     1733
4     3932
5     1116
6     1120
7     2242
8     2029
9     1506
10     790
11    2059
Name: count, dtype: int64

In [93]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 3),
    min_df=10,
    max_features=5000
)

X_tfidf = vectorizer.fit_transform(df["ticket_text"].astype(str))
terms = np.array(vectorizer.get_feature_names_out())

In [94]:
def get_top_terms_for_cluster(cluster_id, top_n=10):
    mask = df["semantic_cluster"] == cluster_id
    
    cluster_tfidf = X_tfidf[mask].mean(axis=0)
    cluster_tfidf = np.asarray(cluster_tfidf).ravel()
    
    top_indices = cluster_tfidf.argsort()[::-1][:top_n]
    
    return terms[top_indices].tolist()

In [95]:
cluster_rows = []

for cluster_id in range(n_clusters):
    mask = df["semantic_cluster"] == cluster_id
    
    sample_subjects = (
        df.loc[mask, "Ticket_Subject"]
        .sample(min(5, mask.sum()), random_state=42)
        .tolist()
    )
    
    cluster_rows.append({
        "cluster": cluster_id,
        "size": int(mask.sum()),
        "top_terms": get_top_terms_for_cluster(cluster_id, top_n=10),
        "sample_subjects": sample_subjects
    })

cluster_summary = pd.DataFrame(cluster_rows)
cluster_summary

,cluster,size,top_terms,sample_subjects
0,0,1175,"[log, account resetting password, hi support l...","[Login failed - Beyond, API Error 500 - Pressu..."
1,1,1245,"[invoice, send invoice transaction, 99283, sup...","[Invoice discrepancy - Reduce, Charged twice -..."
2,2,1053,"[11, patch windows 11, hi support install, ins...","[API Error 500 - Bill, Screen freezes - True, ..."
3,3,1733,"[charge, support higher agreed, hi support hig...","[Invoice discrepancy - Star, Invoice discrepan..."
4,4,3932,"[request, demo, pricing, support, team, hours,...","[Product question - Gun, Office location - Pay..."
5,5,1116,"[api, 500, error, internal server, endpoint, a...","[Data not syncing - Customer, App crashing - R..."
6,6,1120,"[tab, time open settings, application crashes ...","[App crashing - Already, API Error 500 - Own, ..."
7,7,2242,"[email, change email, password reset, reset, c...","[2FA issues - Organization, Subscription upgra..."
8,8,2029,"[2fa, pass 2fa check, pass 2fa, support lost p...","[Account hacked - Result, Subscription upgrade..."
9,9,1506,"[data, spinning, support dashboard loading, lo...","[App crashing - Bill, App crashing - Gas, Inst..."


In [96]:
def show_cluster(cluster_id, n=8):
    cols = [
        "Ticket_ID",
        "Ticket_Subject",
        "Ticket_Description",
        "Issue_Category",
        "Ticket_Channel",
        "Resolution_Time_Hours",
        "rule_severity_score",
        "rule_evidence"
    ]
    
    display(
        df[df["semantic_cluster"] == cluster_id][cols]
        .sample(n, random_state=42)
    )

In [105]:
show_cluster(3)

,Ticket_ID,Ticket_Subject,Ticket_Description,Issue_Category,Ticket_Channel,Resolution_Time_Hours,rule_severity_score,rule_evidence
18989,TKT-118989,Invoice discrepancy - Star,"Hi Support, I noticed a double charge on my st...",Billing,Email,25,0,[]
5923,TKT-105923,Invoice discrepancy - Win,"Hi Support, I noticed a double charge on my st...",Billing,Web Form,17,0,[]
10717,TKT-110717,Update credit card - Put,"Hi Support, I noticed a double charge on my st...",Billing,Chat,35,0,[]
17281,TKT-117281,Refund status - Letter,"Hi Support, Why is my bill higher than the agr...",Billing,Email,31,0,[]
19277,TKT-119277,Update credit card - Letter,"Hi Support, Why is my bill higher than the agr...",Billing,Web Form,2,0,[]
14833,TKT-114833,Refund status - Maybe,"Hi Support, I noticed a double charge on my st...",Billing,Web Form,15,0,[]
405,TKT-100405,Refund status - Mother,"Hi Support, Why is my bill higher than the agr...",Billing,Chat,120,0,[]
14264,TKT-114264,Suspicious charge - Here,"Hi Support, Why is my bill higher than the agr...",Billing,Email,120,3,"[{'signal': 'keyword', 'rule_name': 'suspiciou..."


In [106]:
cluster_severity_map = {
    0: 2,   # technical mixed: login/API/data/app issues
    1: 2,   # billing: invoice discrepancy, charged twice, payment failed
    2: 0,   # Windows patch / latest patch / install info
    3: 2,   # billing dispute: double charge, refund, suspicious charge
    4: 0,   # general inquiry: demo, pricing, office hours, product question
    5: 3,   # API 500 / internal server error
    6: 2,   # app crash / settings tab / freeze
    7: 1,   # change email / password reset
    8: 3,   # 2FA / account hacked / access security
    9: 2,   # dashboard loading / data sync / spinning wheel
    10: 1,  # delete account
    11: 2   # refund / payment / credit card / suspicious charge
}

df["semantic_severity_score"] = df["semantic_cluster"].map(cluster_severity_map)

In [107]:
df["semantic_severity_score"].value_counts()

semantic_severity_score
2    8838
0    4985
3    3145
1    3032
Name: count, dtype: int64

In [108]:
pd.crosstab(
    df["rule_severity_score"],
    df["semantic_severity_score"],
    normalize="index"
) * 100

semantic_severity_score,0,1,2,3
rule_severity_score,,,,
0,38.248188,9.555337,40.824908,11.371567
1,5.535324,71.522214,6.190823,16.751639
2,14.789303,0.000000,85.210697,0.000000
3,5.347742,0.000000,55.472430,39.179828


In [109]:
semantic_exact_agreement = (
    df["rule_severity_score"] == df["semantic_severity_score"]
).mean()

semantic_relaxed_agreement = (
    abs(df["rule_severity_score"] - df["semantic_severity_score"]) <= 1
).mean()

print( semantic_exact_agreement)
print(semantic_relaxed_agreement)

0.4878
0.6574


In [110]:
def create_fusion_score(df, rule_w, semantic_w, resolution_w):
    score = np.round(
        rule_w * df["rule_severity_score"] +
        semantic_w * df["semantic_severity_score"] +
        resolution_w * df["resolution_signal_score"]
    ).astype(int)
    
    return score.clip(0, 3)

In [111]:
fusion_configs = {
    "rule_heavy": (0.55, 0.35, 0.10),
    "balanced_text": (0.45, 0.45, 0.10),
    "semantic_heavy": (0.35, 0.55, 0.10),
    "text_only": (0.50, 0.50, 0.00)
}

In [112]:
for name, weights in fusion_configs.items():
    rule_w, semantic_w, resolution_w = weights
    
    inferred = create_fusion_score(df, rule_w, semantic_w, resolution_w)
    delta = inferred - df["assigned_priority_score"]
    mismatch = (delta.abs() >= 2).astype(int)
    
    def mismatch_type(d):
        if d >= 2:
            return "Hidden Crisis"
        elif d <= -2:
            return "False Alarm"
        else:
            return "Consistent"
    
    types = delta.apply(mismatch_type)
    
    print("\n", name)
    print("Weights:", weights)
    print("Inferred severity distribution:")
    print(inferred.value_counts().sort_index())
    print("\nMismatch label distribution:")
    print(mismatch.value_counts())
    print("\nMismatch type distribution:")
    print(types.value_counts())
    print("-" * 80)


 rule_heavy
Weights: (0.55, 0.35, 0.1)
Inferred severity distribution:
0    4800
1    8988
2    3835
3    2377
Name: count, dtype: int64

Mismatch label distribution:
0    16362
1     3638
Name: count, dtype: int64

Mismatch type distribution:
Consistent       16362
Hidden Crisis     2464
False Alarm       1174
Name: count, dtype: int64
--------------------------------------------------------------------------------

 balanced_text
Weights: (0.45, 0.45, 0.1)
Inferred severity distribution:
0    4570
1    8622
2    4892
3    1916
Name: count, dtype: int64

Mismatch label distribution:
0    16622
1     3378
Name: count, dtype: int64

Mismatch type distribution:
Consistent       16622
Hidden Crisis     2415
False Alarm        963
Name: count, dtype: int64
--------------------------------------------------------------------------------

 semantic_heavy
Weights: (0.35, 0.55, 0.1)
Inferred severity distribution:
0    4350
1    8269
2    5967
3    1414
Name: count, dtype: int64

Mismatch lab

In [113]:
df["inferred_severity_score"] = np.round(
    0.45 * df["rule_severity_score"] +
    0.45 * df["semantic_severity_score"] +
    0.10 * df["resolution_signal_score"]
).astype(int)

df["inferred_severity_score"] = df["inferred_severity_score"].clip(0, 3)

df["severity_delta"] = df["inferred_severity_score"] - df["assigned_priority_score"]

df["mismatch_label"] = (df["severity_delta"].abs() >= 2).astype(int)

def get_mismatch_type(delta):
    if delta >= 2:
        return "Hidden Crisis"
    elif delta <= -2:
        return "False Alarm"
    else:
        return "Consistent"

df["mismatch_type"] = df["severity_delta"].apply(get_mismatch_type)

In [114]:
df[df["mismatch_type"] == "Hidden Crisis"][
    [
        "Ticket_ID",
        "Priority_Level",
        "Ticket_Subject",
        "Ticket_Description",
        "rule_severity_score",
        "semantic_severity_score",
        "resolution_signal_score",
        "inferred_severity_score",
        "severity_delta",
        "rule_evidence"
    ]
].sample(10, random_state=42)

,Ticket_ID,Priority_Level,Ticket_Subject,Ticket_Description,rule_severity_score,semantic_severity_score,resolution_signal_score,inferred_severity_score,severity_delta,rule_evidence
3435,TKT-103435,Low,Installation issue - Accept,"Hi Support, We are receiving a 500 Internal Se...",3,3,2,3,3,"[{'signal': 'keyword', 'rule_name': 'api_serve..."
1623,TKT-101623,Low,Installation issue - Plan,"Hi Support, The application crashes every time...",2,2,3,2,2,"[{'signal': 'keyword', 'rule_name': 'applicati..."
13920,TKT-113920,Low,Data not syncing - Federal,"Hi Support, My data hasn't synced to the cloud...",2,2,2,2,2,"[{'signal': 'keyword', 'rule_name': 'data_sync..."
16134,TKT-116134,Medium,App crashing - Natural,"Hi Support, We are receiving a 500 Internal Se...",3,3,2,3,2,"[{'signal': 'keyword', 'rule_name': 'api_serve..."
8685,TKT-108685,Low,Login failed - Could,"Hi Support, I cannot log into my account even ...",2,2,2,2,2,"[{'signal': 'keyword', 'rule_name': 'login_fai..."
19735,TKT-119735,Low,Screen freezes - Television,"Hi Support, The dashboard is not loading any d...",2,2,2,2,2,"[{'signal': 'keyword', 'rule_name': 'applicati..."
2053,TKT-102053,Medium,Suspicious charge - Itself,"Hi Support, My subscription renewed automatica...",3,2,3,3,2,"[{'signal': 'keyword', 'rule_name': 'suspiciou..."
12291,TKT-112291,Low,Screen freezes - Million,"Hi Support, The application crashes every time...",2,2,2,2,2,"[{'signal': 'keyword', 'rule_name': 'applicati..."
12876,TKT-112876,Low,Login failed - Apply,"Hi Support, We are receiving a 500 Internal Se...",3,3,3,3,3,"[{'signal': 'keyword', 'rule_name': 'api_serve..."
17289,TKT-117289,Low,Screen freezes - Short,"Hi Support, My data hasn't synced to the cloud...",3,2,1,2,2,"[{'signal': 'keyword', 'rule_name': 'applicati..."


In [115]:
df[df["mismatch_type"] == "False Alarm"][
    [
        "Ticket_ID",
        "Priority_Level",
        "Ticket_Subject",
        "Ticket_Description",
        "rule_severity_score",
        "semantic_severity_score",
        "resolution_signal_score",
        "inferred_severity_score",
        "severity_delta",
        "rule_evidence"
    ]
].sample(10, random_state=42)

,Ticket_ID,Priority_Level,Ticket_Subject,Ticket_Description,rule_severity_score,semantic_severity_score,resolution_signal_score,inferred_severity_score,severity_delta,rule_evidence
10019,TKT-110019,High,Product question - Daughter,"Hi Support, What are your support operating ho...",0,0,1,0,-2,[]
6037,TKT-106037,High,Hours of operation - Food,"Hi Support, Is there a roadmap for new feature...",0,0,0,0,-2,[]
6260,TKT-106260,High,Product question - Contain,"Hi Support, Is there a roadmap for new feature...",0,0,0,0,-2,[]
16706,TKT-116706,High,Product question - Democratic,"Hi Support, How does the team pricing tier wor...",0,0,0,0,-2,[]
4889,TKT-104889,High,Profile update - Home,"Hi Support, My profile picture is not updating...",0,1,0,0,-2,[]
13422,TKT-113422,Critical,Alert notification - Edge,"Hi Support, Why is there a new device added to...",0,3,0,1,-2,[]
3671,TKT-103671,Critical,Stolen card - Almost,"Hi Support, I think my account has been compro...",0,3,0,1,-2,[]
9355,TKT-109355,High,Pricing tiers - Open,"Hi Support, Where is your headquarters located...",0,0,0,0,-2,[]
1657,TKT-101657,Critical,Phishing attempt - Prove,"Hi Support, I think my account has been compro...",0,3,0,1,-2,[]
5062,TKT-105062,High,Hours of operation - Region,"Hi Support, How does the team pricing tier wor...",0,0,1,0,-2,[]


In [118]:
extra= [
    {
        "rule_name": "account_security_risk",
        "severity_score": 3,
        "severity_label": "critical",
        "weight": 4,
        "phrases": [
            "account has been compromised",
            "account compromised",
            "phishing attempt",
            "stolen card",
            "new device added",
            "account hacked",
            "unauthorized access"
        ]
    },
    {
        "rule_name": "billing_payment_risk",
        "severity_score": 2,
        "severity_label": "high",
        "weight": 3,
        "phrases": [
            "payment failed",
            "charged twice",
            "double charge",
            "suspicious charge",
            "bill higher",
            "invoice discrepancy",
            "refund status"
        ]
    }
]

severity_rules.extend(extra)

In [119]:
rule_results = df["ticket_text"].apply(rule_based_severity_v2)

df["rule_score"] = rule_results.apply(lambda x: x[0])
df["rule_severity_score"] = rule_results.apply(lambda x: x[1])
df["rule_evidence"] = rule_results.apply(lambda x: x[2])

In [120]:
df["rule_severity_score"].value_counts()

rule_severity_score
0    6931
2    6128
3    4195
1    2746
Name: count, dtype: int64

In [128]:
base_score = np.round(
    0.45 * df["rule_severity_score"] +
    0.45 * df["semantic_severity_score"] +
    0.10 * df["resolution_signal_score"]
).astype(int)

base_score = base_score.clip(0, 3)

df["inferred_severity_score"] = base_score

# safeguard 1: if semantic cluster says Critical, do not allow final severity below High
df.loc[
    (df["semantic_severity_score"] == 3) &
    (df["inferred_severity_score"] < 2),
    "inferred_severity_score"
] = 2

# safeguard 2: if both rule and semantic are serious, keep it serious
df.loc[
    (df["rule_severity_score"] >= 2) &
    (df["semantic_severity_score"] >= 2) &
    (df["inferred_severity_score"] < 2),
    "inferred_severity_score"
] = 2

# safeguard 3: if rule is critical and semantic is at least high, keep critical
df.loc[
    (df["rule_severity_score"] == 3) &
    (df["semantic_severity_score"] >= 2),
    "inferred_severity_score"
] = 3

In [129]:
df["severity_delta"] = df["inferred_severity_score"] - df["assigned_priority_score"]

df["mismatch_label"] = (df["severity_delta"].abs() >= 2).astype(int)

def get_mismatch_type(delta):
    if delta >= 2:
        return "Hidden Crisis"
    elif delta <= -2:
        return "False Alarm"
    else:
        return "Consistent"

df["mismatch_type"] = df["severity_delta"].apply(get_mismatch_type)

In [130]:
print(df["inferred_severity_score"].value_counts())
print(df["mismatch_label"].value_counts())
print(df["mismatch_type"].value_counts())

inferred_severity_score
2    7038
0    4570
1    4276
3    4116
Name: count, dtype: int64
mismatch_label
0    14437
1     5563
Name: count, dtype: int64
mismatch_type
Consistent       14437
Hidden Crisis     4911
False Alarm        652
Name: count, dtype: int64


In [131]:
df[df["mismatch_type"] == "False Alarm"][
    [
        "Ticket_ID",
        "Priority_Level",
        "Ticket_Subject",
        "Ticket_Description",
        "rule_severity_score",
        "semantic_severity_score",
        "resolution_signal_score",
        "inferred_severity_score",
        "severity_delta",
        "rule_evidence"
    ]
].sample(10, random_state=42)

,Ticket_ID,Priority_Level,Ticket_Subject,Ticket_Description,rule_severity_score,semantic_severity_score,resolution_signal_score,inferred_severity_score,severity_delta,rule_evidence
19628,TKT-119628,High,Demo request - Under,"Hi Support, Where is your headquarters located...",0,0,3,0,-2,[]
7062,TKT-107062,High,Pricing tiers - Toward,"Hi Support, Where is your headquarters located...",0,0,1,0,-2,[]
6975,TKT-106975,High,Hours of operation - Year,"Hi Support, How does the team pricing tier wor...",0,0,1,0,-2,[]
6039,TKT-106039,High,Product question - Difficult,"Hi Support, I would like to request a demo for...",0,0,3,0,-2,[]
1879,TKT-101879,High,Subscription upgrade - Capital,"Hi Support, I want to permanently delete my ac...",0,1,0,0,-2,[]
14147,TKT-114147,High,Feature request - Evening,"Hi Support, How does the team pricing tier wor...",0,0,1,0,-2,[]
11626,TKT-111626,High,Product question - Near,"Hi Support, What are your support operating ho...",0,0,1,0,-2,[]
3241,TKT-103241,High,Product question - Direction,"Hi Support, Where is your headquarters located...",0,0,0,0,-2,[]
14467,TKT-114467,High,Data not syncing - Drug,"Hi Support, How do I install the latest patch ...",0,0,3,0,-2,"[{'signal': 'keyword', 'rule_name': 'patch_inf..."
19375,TKT-119375,High,Pricing tiers - Main,"Hi Support, Where is your headquarters located...",0,0,1,0,-2,[]


In [126]:
security_rule = {
    "rule_name": "suspicious_login_or_lock_request",
    "severity_score": 3,
    "severity_label": "critical",
    "weight": 4,
    "phrases": [
        "unrecognized login",
        "lock my account",
        "lock my account immediately",
        "unknown login",
        "new device added",
        "unknown device",
        "device added",
        "account has been compromised",
        "account compromised"
    ]
}

severity_rules.append(security_rule)

In [127]:
rule_results = df["ticket_text"].apply(rule_based_severity_v2)

df["rule_score"] = rule_results.apply(lambda x: x[0])
df["rule_severity_score"] = rule_results.apply(lambda x: x[1])
df["rule_evidence"] = rule_results.apply(lambda x: x[2])

In [132]:
df.loc[
    ((df["rule_severity_score"] >= 1) | (df["semantic_severity_score"] >= 1)) &
    (df["inferred_severity_score"] < 1),
    "inferred_severity_score"
] = 1

In [133]:
df["severity_delta"] = df["inferred_severity_score"] - df["assigned_priority_score"]
df["mismatch_label"] = (df["severity_delta"].abs() >= 2).astype(int)

def get_mismatch_type(delta):
    if delta >= 2:
        return "Hidden Crisis"
    elif delta <= -2:
        return "False Alarm"
    else:
        return "Consistent"

df["mismatch_type"] = df["severity_delta"].apply(get_mismatch_type)

print(df["inferred_severity_score"].value_counts().sort_index())
print(df["mismatch_label"].value_counts())
print(df["mismatch_type"].value_counts())

inferred_severity_score
0    4275
1    4571
2    7038
3    4116
Name: count, dtype: int64
mismatch_label
0    14482
1     5518
Name: count, dtype: int64
mismatch_type
Consistent       14482
Hidden Crisis     4911
False Alarm        607
Name: count, dtype: int64


In [134]:
df[df["mismatch_type"] == "False Alarm"][
    [
        "Ticket_ID",
        "Priority_Level",
        "Ticket_Subject",
        "Ticket_Description",
        "rule_severity_score",
        "semantic_severity_score",
        "resolution_signal_score",
        "inferred_severity_score",
        "severity_delta",
        "rule_evidence"
    ]
].sample(10, random_state=42)

,Ticket_ID,Priority_Level,Ticket_Subject,Ticket_Description,rule_severity_score,semantic_severity_score,resolution_signal_score,inferred_severity_score,severity_delta,rule_evidence
18374,TKT-118374,High,Hours of operation - Town,"Hi Support, Where is your headquarters located...",0,0,3,0,-2,[]
8350,TKT-108350,High,Hours of operation - Read,"Hi Support, How does the team pricing tier wor...",0,0,0,0,-2,[]
2057,TKT-102057,High,Demo request - List,"Hi Support, I would like to request a demo for...",0,0,2,0,-2,[]
2103,TKT-102103,Critical,Screen freezes - Husband,"Hi Support, How do I install the latest patch ...",2,0,1,1,-2,"[{'signal': 'keyword', 'rule_name': 'applicati..."
5005,TKT-105005,Critical,App crashing - Machine,"Hi Support, How do I install the latest patch ...",0,0,0,0,-3,"[{'signal': 'keyword', 'rule_name': 'patch_inf..."
15562,TKT-115562,High,Data not syncing - Course,"Hi Support, How do I install the latest patch ...",0,0,1,0,-2,"[{'signal': 'keyword', 'rule_name': 'patch_inf..."
351,TKT-100351,High,Data not syncing - Range,"Hi Support, How do I install the latest patch ...",0,0,0,0,-2,"[{'signal': 'keyword', 'rule_name': 'patch_inf..."
3507,TKT-103507,High,Data not syncing - Page,"Hi Support, How do I install the latest patch ...",0,0,3,0,-2,"[{'signal': 'keyword', 'rule_name': 'patch_inf..."
14055,TKT-114055,High,Demo request - Eat,"Hi Support, How does the team pricing tier wor...",0,0,1,0,-2,[]
2250,TKT-102250,Critical,Data not syncing - Low,"Hi Support, I cannot log into my account even ...",0,2,1,1,-2,[]


In [135]:
pd.crosstab(
    df["Priority_Level"],
    df["mismatch_type"]
)

mismatch_type,Consistent,False Alarm,Hidden Crisis
Priority_Level,,,
Critical,1180,118,0
High,2927,489,0
Low,4069,0,3647
Medium,6306,0,1264


In [136]:
pd.crosstab(
    df["Priority_Level"],
    df["mismatch_type"],
    normalize="index"
) * 100

mismatch_type,Consistent,False Alarm,Hidden Crisis
Priority_Level,,,
Critical,90.909091,9.090909,0.000000
High,85.685012,14.314988,0.000000
Low,52.734578,0.000000,47.265422
Medium,83.302510,0.000000,16.697490


In [137]:
df["severity_delta"].value_counts()

severity_delta
 1    5915
 0    5909
 2    4053
-1    2658
 3     858
-2     583
-3      24
Name: count, dtype: int64

In [138]:
df.loc[
    (df["rule_severity_score"] >= 2) &
    (df["inferred_severity_score"] < 2),
    "inferred_severity_score"
] = 2

In [139]:
df["severity_delta"] = df["inferred_severity_score"] - df["assigned_priority_score"]
df["mismatch_label"] = (df["severity_delta"].abs() >= 2).astype(int)

def get_mismatch_type(delta):
    if delta >= 2:
        return "Hidden Crisis"
    elif delta <= -2:
        return "False Alarm"
    else:
        return "Consistent"

df["mismatch_type"] = df["severity_delta"].apply(get_mismatch_type)

In [140]:
pd.crosstab(
    df["Priority_Level"],
    df["mismatch_type"],
    normalize="index"
) * 100

mismatch_type,Consistent,False Alarm,Hidden Crisis
Priority_Level,,,
Critical,94.298921,5.701079,0.000000
High,85.685012,14.314988,0.000000
Low,51.477449,0.000000,48.522551
Medium,83.302510,0.000000,16.697490


In [141]:
pd.crosstab(
    df["Priority_Level"],
    df["mismatch_type"]
)

mismatch_type,Consistent,False Alarm,Hidden Crisis
Priority_Level,,,
Critical,1224,74,0
High,2927,489,0
Low,3972,0,3744
Medium,6306,0,1264


In [142]:
print(df["inferred_severity_score"].value_counts().sort_index())
print(df["mismatch_label"].value_counts())
print(df["mismatch_type"].value_counts())

inferred_severity_score
0    4275
1    4114
2    7495
3    4116
Name: count, dtype: int64
mismatch_label
0    14429
1     5571
Name: count, dtype: int64
mismatch_type
Consistent       14429
Hidden Crisis     5008
False Alarm        563
Name: count, dtype: int64


In [143]:
df[
    (df["Priority_Level"] == "Low") &
    (df["mismatch_type"] == "Hidden Crisis")
][
    [
        "Ticket_ID",
        "Priority_Level",
        "Ticket_Subject",
        "Ticket_Description",
        "rule_severity_score",
        "semantic_severity_score",
        "resolution_signal_score",
        "inferred_severity_score",
        "severity_delta",
        "rule_evidence"
    ]
].sample(15, random_state=7)

,Ticket_ID,Priority_Level,Ticket_Subject,Ticket_Description,rule_severity_score,semantic_severity_score,resolution_signal_score,inferred_severity_score,severity_delta,rule_evidence
14972,TKT-114972,Low,Password reset - Bank,"Hi Support, I lost my phone and cannot pass th...",1,3,1,2,2,"[{'signal': 'keyword', 'rule_name': 'account_a..."
7547,TKT-107547,Low,Update credit card - Training,"Hi Support, I noticed a double charge on my st...",2,2,0,2,2,"[{'signal': 'keyword', 'rule_name': 'billing_p..."
1679,TKT-101679,Low,2FA issues - Commercial,"Hi Support, I lost my phone and cannot pass th...",0,3,3,2,2,[]
13718,TKT-113718,Low,Invoice discrepancy - Strong,"Hi Support, I requested a refund 5 days ago, w...",2,2,1,2,2,"[{'signal': 'keyword', 'rule_name': 'billing_p..."
2508,TKT-102508,Low,Login failed - Suggest,"Hi Support, My data hasn't synced to the cloud...",3,2,3,3,3,"[{'signal': 'keyword', 'rule_name': 'login_fai..."
16551,TKT-116551,Low,Screen freezes - Article,"Hi Support, My data hasn't synced to the cloud...",3,2,1,3,3,"[{'signal': 'keyword', 'rule_name': 'applicati..."
12927,TKT-112927,Low,Refund status - Probably,"Hi Support, I noticed a double charge on my st...",2,2,2,2,2,"[{'signal': 'keyword', 'rule_name': 'billing_p..."
3775,TKT-103775,Low,Profile update - Person,"Hi Support, I lost my phone and cannot pass th...",0,3,1,2,2,[]
19364,TKT-119364,Low,Invoice discrepancy - Front,"Hi Support, Can you send me the invoice for tr...",2,2,1,2,2,"[{'signal': 'keyword', 'rule_name': 'billing_p..."
6784,TKT-106784,Low,Invoice discrepancy - Build,"Hi Support, My subscription renewed automatica...",2,2,3,2,2,"[{'signal': 'keyword', 'rule_name': 'billing_p..."


In [144]:
df.to_csv("../data/processed/sia_pseudo_labeled_v1.csv", index=False)

In [145]:
print(df.shape)
print(df["mismatch_label"].value_counts())
print(df["mismatch_type"].value_counts())

(20000, 26)
mismatch_label
0    14429
1     5571
Name: count, dtype: int64
mismatch_type
Consistent       14429
Hidden Crisis     5008
False Alarm        563
Name: count, dtype: int64


In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/sia_pseudo_labeled_v1.csv")

In [2]:
df.head()

,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,...,rule_evidence,resolution_time_score,resolution_percentile,resolution_signal_score,semantic_cluster,semantic_severity_score,inferred_severity_score,severity_delta,mismatch_label,mismatch_type
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",General Inquiry,High,Web Form,2025-07-02,43,...,[],2,0.617871,2,4,0,0,-2,1,False Alarm
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,...,"[{'signal': 'keyword', 'rule_name': 'applicati...",2,0.680307,2,6,2,2,0,0,Consistent
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,...,[],0,0.150746,0,8,3,2,0,0,Consistent
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,...,"[{'signal': 'keyword', 'rule_name': 'applicati...",2,0.680070,2,9,2,3,3,1,Hidden Crisis
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,...,"[{'signal': 'keyword', 'rule_name': 'billing_p...",2,0.589713,2,11,2,2,1,0,Consistent


In [3]:
type(df.loc[1, "rule_evidence"])

str

In [4]:
import ast
def safe_rule_evidence(value):
    if isinstance(value, list):
        return value
    
    if pd.isna(value):
        return []
    
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except:
            return []
    
    return []

df["rule_evidence"] = df["rule_evidence"].apply(safe_rule_evidence)

In [5]:
type(df.loc[1, "rule_evidence"])

list

In [6]:
severity_label_map = {
    0: "Low",
    1: "Medium",
    2: "High",
    3: "Critical"
}

In [7]:
def short_text(text, max_len=180):
    text = str(text)
    if len(text) <= max_len:
        return text
    return text[:max_len] + "..."

In [8]:
def generate_evidence_dossier(row):
    assigned_priority = row["Priority_Level"]
    inferred_score = int(row["inferred_severity_score"])
    inferred_severity = severity_label_map[inferred_score]
    severity_delta = int(row["severity_delta"])
    mismatch_type = row["mismatch_type"]

    feature_evidence = []

    # keyword evidence
    rule_items = row["rule_evidence"]

    if isinstance(rule_items, list) and len(rule_items) > 0:
        for item in rule_items:
            feature_evidence.append({
                "signal": "keyword",
                "value": item.get("value", ""),
                "weight": item.get("weight", ""),
                "interpretation": f"Matched rule: {item.get('rule_name', '')}"
            })

    #  Subject evidence
    feature_evidence.append({
        "signal": "ticket_subject",
        "value": str(row["Ticket_Subject"]),
        "interpretation": "Ticket subject used as direct text evidence."
    })

    #  Description evidence
    feature_evidence.append({
        "signal": "ticket_description",
        "value": short_text(row["Ticket_Description"]),
        "interpretation": "Ticket description used as direct text evidence."
    })

    #  Resolution evidence
    resolution_percentile = row["resolution_percentile"]

    if resolution_percentile >= 0.75:
        resolution_interpretation = "Resolution time is slower than most similar tickets."
    elif resolution_percentile <= 0.25:
        resolution_interpretation = "Resolution time is faster than most similar tickets."
    else:
        resolution_interpretation = "Resolution time is in the middle range among similar tickets."

    feature_evidence.append({
        "signal": "resolution_time",
        "value": f"{row['Resolution_Time_Hours']} hours",
        "interpretation": resolution_interpretation
    })

    #  Metadata evidence
    feature_evidence.append({
        "signal": "metadata",
        "value": f"Issue category: {row['Issue_Category']}, Channel: {row['Ticket_Channel']}",
        "interpretation": "Structured metadata used as context."
    })

    signal_scores = [
        int(row["rule_severity_score"]),
        int(row["semantic_severity_score"]),
        int(row["resolution_signal_score"])
    ]

    close_votes = sum(abs(score - inferred_score) <= 1 for score in signal_scores)

    if abs(severity_delta) == 3 and close_votes >= 2:
        confidence = "High"
    elif close_votes >= 2:
        confidence = "Medium"
    else:
        confidence = "Low"

    if mismatch_type == "Hidden Crisis":
        constraint_analysis = (
            f"The ticket was assigned {assigned_priority}, but the inferred severity is {inferred_severity}. "
            f"The severity delta is {severity_delta}, so the ticket is flagged as Hidden Crisis because the inferred "
            f"severity is at least two levels higher than the assigned priority."
        )

    elif mismatch_type == "False Alarm":
        constraint_analysis = (
            f"The ticket was assigned {assigned_priority}, but the inferred severity is {inferred_severity}. "
            f"The severity delta is {severity_delta}, so the ticket is flagged as False Alarm because the inferred "
            f"severity is at least two levels lower than the assigned priority."
        )

    else:
        constraint_analysis = (
            f"The ticket was assigned {assigned_priority}, and the inferred severity is {inferred_severity}. "
            f"The difference is not large enough to flag it as a priority mismatch."
        )

    dossier = {
        "ticket_id": row["Ticket_ID"],
        "assigned_priority": assigned_priority,
        "inferred_severity": inferred_severity,
        "mismatch_type": mismatch_type,
        "severity_delta": severity_delta,
        "feature_evidence": feature_evidence,
        "constraint_analysis": constraint_analysis,
        "confidence": confidence
    }

    return dossier

In [9]:
df["evidence_dossier"] = None

flagged_mask = df["mismatch_label"] == 1

df.loc[flagged_mask, "evidence_dossier"] = df[flagged_mask].apply(
    generate_evidence_dossier,
    axis=1
)

In [11]:
import json
sample_hidden = df[df["mismatch_type"] == "Hidden Crisis"].sample(1, random_state=42)
hidden_dossier = sample_hidden.iloc[0]["evidence_dossier"]

print(json.dumps(hidden_dossier, indent=4))

{
    "ticket_id": "TKT-109854",
    "assigned_priority": "Low",
    "inferred_severity": "Critical",
    "mismatch_type": "Hidden Crisis",
    "severity_delta": 3,
    "feature_evidence": [
        {
            "signal": "keyword",
            "value": "suspicious",
            "weight": 4,
            "interpretation": "Matched rule: suspicious_activity"
        },
        {
            "signal": "keyword",
            "value": "suspicious charge",
            "weight": 3,
            "interpretation": "Matched rule: billing_payment_risk"
        },
        {
            "signal": "ticket_subject",
            "value": "Suspicious charge - Provide",
            "interpretation": "Ticket subject used as direct text evidence."
        },
        {
            "signal": "ticket_description",
            "value": "Hi Support, My subscription renewed automatically, but I wanted to cancel. Blue improve general.",
            "interpretation": "Ticket description used as direct text eviden

In [12]:
sample_false = df[df["mismatch_type"] == "False Alarm"].sample(1, random_state=42)
false_alarm_dossier = sample_false.iloc[0]["evidence_dossier"]

print(json.dumps(false_alarm_dossier, indent=4))

{
    "ticket_id": "TKT-107630",
    "assigned_priority": "High",
    "inferred_severity": "Low",
    "mismatch_type": "False Alarm",
    "severity_delta": -2,
    "feature_evidence": [
        {
            "signal": "ticket_subject",
            "value": "Demo request - Certainly",
            "interpretation": "Ticket subject used as direct text evidence."
        },
        {
            "signal": "ticket_description",
            "value": "Hi Support, How does the team pricing tier work? Top brother sign animal.",
            "interpretation": "Ticket description used as direct text evidence."
        },
        {
            "signal": "resolution_time",
            "value": "8 hours",
            "interpretation": "Resolution time is faster than most similar tickets."
        },
        {
            "signal": "metadata",
            "value": "Issue category: General Inquiry, Channel: Web Form",
            "interpretation": "Structured metadata used as context."
        }
    ],

In [14]:
df.head(1)

,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,...,resolution_time_score,resolution_percentile,resolution_signal_score,semantic_cluster,semantic_severity_score,inferred_severity_score,severity_delta,mismatch_label,mismatch_type,evidence_dossier
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",General Inquiry,High,Web Form,2025-07-02,43,...,2,0.617871,2,4,0,0,-2,1,False Alarm,"{'ticket_id': 'TKT-100000', 'assigned_priority..."


In [15]:
def clean_keyword_evidence(rule_items):

    if not isinstance(rule_items, list):
        return []

    valid_items = []

    for item in rule_items:
        if isinstance(item, dict) and item.get("value", ""):
            valid_items.append(item)

    cleaned_items = []

    for item in valid_items:
        keyword = str(item.get("value", "")).lower().strip()

        is_generic_duplicate = False

        for other in valid_items:
            other_keyword = str(other.get("value", "")).lower().strip()

            if keyword == other_keyword:
                continue

            # If current keyword is inside a longer keyword, skip current one
            if keyword in other_keyword and len(other_keyword) > len(keyword):
                is_generic_duplicate = True
                break

        if not is_generic_duplicate:
            cleaned_items.append(item)

    return cleaned_items

In [24]:
def generate_evidence_dossier(row):
    assigned_priority = row["Priority_Level"]
    inferred_score = int(row["inferred_severity_score"])
    inferred_severity = severity_label_map[inferred_score]
    severity_delta = int(row["severity_delta"])
    mismatch_type = row["mismatch_type"]

    feature_evidence = []

   # keyword evidence
    rule_items = clean_keyword_evidence(row["rule_evidence"])
    
    for item in rule_items:
        feature_evidence.append({
            "signal": "keyword",
            "value": item.get("value", ""),
            "weight": item.get("weight", ""),
            "source_field": "Ticket_Subject / Ticket_Description",
            "interpretation": f"Matched rule: {item.get('rule_name', '')}"
        })

    #  Subject evidence
    feature_evidence.append({
        "signal": "ticket_subject",
        "value": str(row["Ticket_Subject"]),
        "source_field": "Ticket_Subject",
        "interpretation": "Ticket subject used as direct text evidence."
    })

    #  Description evidence
    feature_evidence.append({
        "signal": "ticket_description",
        "value": short_text(row["Ticket_Description"]),
        "source_field": "Ticket_Description",
        "interpretation": "Ticket description used as direct text evidence."
    })

    #  Resolution evidence
    resolution_percentile = row["resolution_percentile"]

    if resolution_percentile >= 0.75:
        resolution_interpretation = "Resolution time is slower than most similar tickets."
    elif resolution_percentile <= 0.25:
        resolution_interpretation = "Resolution time is faster than most similar tickets."
    else:
        resolution_interpretation = "Resolution time is in the middle range among similar tickets."

    feature_evidence.append({
        "signal": "resolution_time",
        "value": f"{row['Resolution_Time_Hours']} hours",
        "source_field": "Resolution_Time_Hours",
        "interpretation": resolution_interpretation
    })

    #  Metadata evidence
    feature_evidence.append({
        "signal": "metadata",
        "value": f"Issue category: {row['Issue_Category']}, Channel: {row['Ticket_Channel']}",
         "source_field": "Issue_Category / Ticket_Channel",
        "interpretation": "Structured metadata used as context."
    })

    signal_scores = [
        int(row["rule_severity_score"]),
        int(row["semantic_severity_score"]),
        int(row["resolution_signal_score"])
    ]

    close_votes = sum(abs(score - inferred_score) <= 1 for score in signal_scores)

    if abs(severity_delta) == 3 and close_votes >= 2:
        confidence = "High"
    elif close_votes >= 2:
        confidence = "Medium"
    else:
        confidence = "Low"

    if mismatch_type == "Hidden Crisis":
        constraint_analysis = (
            f"The ticket was assigned {assigned_priority}, but the inferred severity is {inferred_severity}. "
            f"The severity delta is {severity_delta}, so the ticket is flagged as Hidden Crisis because the inferred "
            f"severity is at least two levels higher than the assigned priority."
        )

    elif mismatch_type == "False Alarm":
        constraint_analysis = (
            f"The ticket was assigned {assigned_priority}, but the inferred severity is {inferred_severity}. "
            f"The severity delta is {severity_delta}, so the ticket is flagged as False Alarm because the inferred "
            f"severity is at least two levels lower than the assigned priority."
        )

    else:
        constraint_analysis = (
            f"The ticket was assigned {assigned_priority}, and the inferred severity is {inferred_severity}. "
            f"The difference is not large enough to flag it as a priority mismatch."
        )

    dossier = {
        "ticket_id": row["Ticket_ID"],
        "assigned_priority": assigned_priority,
        "inferred_severity": inferred_severity,
        "mismatch_type": mismatch_type,
        "severity_delta": severity_delta,
        "feature_evidence": feature_evidence,
        "constraint_analysis": constraint_analysis,
        "confidence": confidence
    }

    return dossier

In [25]:
df["evidence_dossier"] = None

flagged_mask = df["mismatch_label"] == 1

df.loc[flagged_mask, "evidence_dossier"] = df[flagged_mask].apply(
    generate_evidence_dossier,
    axis=1
)

In [26]:
sample_hidden = df[df["mismatch_type"] == "Hidden Crisis"].sample(1, random_state=42)
hidden_dossier = sample_hidden.iloc[0]["evidence_dossier"]

print(json.dumps(hidden_dossier, indent=4))

{
    "ticket_id": "TKT-109854",
    "assigned_priority": "Low",
    "inferred_severity": "Critical",
    "mismatch_type": "Hidden Crisis",
    "severity_delta": 3,
    "feature_evidence": [
        {
            "signal": "keyword",
            "value": "suspicious charge",
            "weight": 3,
            "source_field": "Ticket_Subject / Ticket_Description",
            "interpretation": "Matched rule: billing_payment_risk"
        },
        {
            "signal": "ticket_subject",
            "value": "Suspicious charge - Provide",
            "source_field": "Ticket_Subject",
            "interpretation": "Ticket subject used as direct text evidence."
        },
        {
            "signal": "ticket_description",
            "value": "Hi Support, My subscription renewed automatically, but I wanted to cancel. Blue improve general.",
            "source_field": "Ticket_Description",
            "interpretation": "Ticket description used as direct text evidence."
        },
 

In [27]:
sample_false = df[df["mismatch_type"] == "False Alarm"].sample(1, random_state=42)
false_alarm_dossier = sample_false.iloc[0]["evidence_dossier"]

print(json.dumps(false_alarm_dossier, indent=4))

{
    "ticket_id": "TKT-107630",
    "assigned_priority": "High",
    "inferred_severity": "Low",
    "mismatch_type": "False Alarm",
    "severity_delta": -2,
    "feature_evidence": [
        {
            "signal": "ticket_subject",
            "value": "Demo request - Certainly",
            "source_field": "Ticket_Subject",
            "interpretation": "Ticket subject used as direct text evidence."
        },
        {
            "signal": "ticket_description",
            "value": "Hi Support, How does the team pricing tier work? Top brother sign animal.",
            "source_field": "Ticket_Description",
            "interpretation": "Ticket description used as direct text evidence."
        },
        {
            "signal": "resolution_time",
            "value": "8 hours",
            "source_field": "Resolution_Time_Hours",
            "interpretation": "Resolution time is faster than most similar tickets."
        },
        {
            "signal": "metadata",
           

In [28]:
df.to_csv("../data/processed/sia_dossiers.csv", index=False)

In [29]:
flagged_dossiers = df[df["mismatch_label"] == 1]["evidence_dossier"].tolist()

with open("../data/processed/evidence_dossiers_v1.json", "w", encoding="utf-8") as f:
    json.dump(flagged_dossiers, f, indent=4)

In [30]:
 # classifier training 

In [31]:
!pip install transformers accelerate scikit-learn

In [32]:
df["mismatch_label"].value_counts(normalize=True) * 100

mismatch_label
0    72.145
1    27.855
Name: proportion, dtype: float64

In [33]:
df_model = df.copy()

df_model["model_input"] = (
    "Assigned Priority: " + df_model["Priority_Level"].astype(str) +
    " | Issue Category: " + df_model["Issue_Category"].astype(str) +
    " | Ticket Channel: " + df_model["Ticket_Channel"].astype(str) +
    " | Resolution Time Hours: " + df_model["Resolution_Time_Hours"].astype(str) +
    " | Subject: " + df_model["Ticket_Subject"].astype(str) +
    " | Description: " + df_model["Ticket_Description"].astype(str)
)

df_model["label"] = df_model["mismatch_label"].astype(int)

In [34]:
df_model[["model_input", "label"]].head(3)

,model_input,label
0,Assigned Priority: High | Issue Category: Gene...,1
1,Assigned Priority: High | Issue Category: Tech...,0
2,Assigned Priority: High | Issue Category: Acco...,0


In [36]:
from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(
    df_model,
    test_size=0.30,
    random_state=42,
    stratify=df_model["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

In [37]:
print( train_df.shape)
print(val_df.shape)
print(test_df.shape)

(14000, 29)
(3000, 29)
(3000, 29)


In [38]:
print(train_df["label"].value_counts(normalize=True) * 100)
print(val_df["label"].value_counts(normalize=True) * 100)
print(test_df["label"].value_counts(normalize=True) * 100)

label
0    72.142857
1    27.857143
Name: proportion, dtype: float64
label
0    72.133333
1    27.866667
Name: proportion, dtype: float64
label
0    72.166667
1    27.833333
Name: proportion, dtype: float64


In [40]:
from sklearn.utils.class_weight import compute_class_weight
import torch
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print(class_weights)

tensor([0.6931, 1.7949])


In [41]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [42]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

C:\Users\user\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [56]:
class TicketDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=96):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        item = {key: value.squeeze(0) for key, value in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return item

In [57]:
train_dataset = TicketDataset(
    train_df["model_input"],
    train_df["label"],
    tokenizer
)

val_dataset = TicketDataset(
    val_df["model_input"],
    val_df["label"],
    tokenizer
)

test_dataset = TicketDataset(
    test_df["model_input"],
    test_df["label"],
    tokenizer
)

In [58]:
sample_item = train_dataset[0]

print(sample_item.keys())
print(sample_item["input_ids"].shape)
print(sample_item["attention_mask"].shape)
print(sample_item["labels"])

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
torch.Size([96])
torch.Size([96])
tensor(0)


In [59]:
df["description_word_count"] = df_model["model_input"].astype(str).apply(lambda x: len(x.split()))

df["description_word_count"].max()

49

In [60]:
token_lengths = df_model["model_input"].astype(str).apply(
    lambda x: len(tokenizer(x)["input_ids"])
)

token_lengths.describe()
# token_lengths.max()

count    20000.000000
mean        54.115500
std          3.010932
min         46.000000
25%         52.000000
50%         54.000000
75%         56.000000
max         68.000000
Name: model_input, dtype: float64

In [61]:
from transformers import Trainer, TrainingArguments

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )
        
        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )
        
        return (loss, outputs) if return_outputs else loss

In [62]:
from sklearn.metrics import accuracy_score, f1_score, recall_score
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    preds = np.argmax(logits, axis=1)
    
    accuracy = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    recalls = recall_score(labels, preds, average=None, labels=[0, 1])
    
    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "recall_consistent": recalls[0],
        "recall_mismatch": recalls[1]
    }

In [74]:
training_args = TrainingArguments(
    output_dir="../models/sia_distilbert",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    seed=42
)

In [72]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [73]:
trainer.train()

C:\Users\user\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Recall Consistent,Recall Mismatch
1,0.002210,0.023654,0.997333,0.996676,0.999538,0.991627


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1750, training_loss=0.11055013258755207, metrics={'train_runtime': 3856.874, 'train_samples_per_second': 3.63, 'train_steps_per_second': 0.454, 'total_flos': 347726921472000.0, 'train_loss': 0.11055013258755207, 'epoch': 1.0})

In [88]:
import os

model_path = "../models/sia_distilbert_final"


In [89]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

print("Model loaded successfully")
print("Device:", device)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully
Device: cpu


In [90]:
def create_model_input(row):
    return (
        "Assigned Priority: " + str(row["Priority_Level"]) +
        " | Issue Category: " + str(row["Issue_Category"]) +
        " | Ticket Channel: " + str(row["Ticket_Channel"]) +
        " | Resolution Time Hours: " + str(row["Resolution_Time_Hours"]) +
        " | Subject: " + str(row["Ticket_Subject"]) +
        " | Description: " + str(row["Ticket_Description"])
    )

In [91]:
label_map = {
    0: "Consistent",
    1: "Mismatch"
}

def predict_mismatch(row):
    model_input = create_model_input(row)
    
    encoded = tokenizer(
        model_input,
        truncation=True,
        padding="max_length",
        max_length=96,
        return_tensors="pt"
    )
    
    encoded = {key: value.to(device) for key, value in encoded.items()}
    
    with torch.no_grad():
        outputs = model(**encoded)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        predicted_label = int(np.argmax(probs))
    
    return {
        "prediction": predicted_label,
        "prediction_label": label_map[predicted_label],
        "confidence": float(probs[predicted_label]),
        "prob_consistent": float(probs[0]),
        "prob_mismatch": float(probs[1])
    }

In [92]:
sample_row = df.sample(1, random_state=42).iloc[0]

print("Ticket ID:", sample_row["Ticket_ID"])
print("Actual mismatch_label:", sample_row["mismatch_label"])
print("Actual mismatch_type:", sample_row["mismatch_type"])
print("Subject:", sample_row["Ticket_Subject"])
print("Description:", sample_row["Ticket_Description"])

prediction = predict_mismatch(sample_row)

prediction

Ticket ID: TKT-110650
Actual mismatch_label: 1
Actual mismatch_type: Hidden Crisis
Subject: Change email - Trial
Description: Hi Support, I lost my phone and cannot pass the 2FA check. South would four collection trip base provide.


{'prediction': 1,
 'prediction_label': 'Mismatch',
 'confidence': 0.9998664855957031,
 'prob_consistent': 0.0001335397973889485,
 'prob_mismatch': 0.9998664855957031}

In [93]:
custom_ticket = {
    "Priority_Level": "Low",
    "Issue_Category": "Technical",
    "Ticket_Channel": "Chat",
    "Resolution_Time_Hours": 8,
    "Ticket_Subject": "API error 500",
    "Ticket_Description": "Users are unable to access the dashboard because the API returns internal server error."
}

predict_mismatch(custom_ticket)

{'prediction': 1,
 'prediction_label': 'Mismatch',
 'confidence': 0.9998741149902344,
 'prob_consistent': 0.00012584960495587438,
 'prob_mismatch': 0.9998741149902344}

In [94]:
def audit_existing_ticket(row):
    prediction = predict_mismatch(row)
    
    result = {
        "ticket_id": row["Ticket_ID"],
        "model_prediction": prediction["prediction"],
        "model_prediction_label": prediction["prediction_label"],
        "model_confidence": prediction["confidence"],
        "prob_consistent": prediction["prob_consistent"],
        "prob_mismatch": prediction["prob_mismatch"]
    }
    
    if prediction["prediction"] == 1:
        result["dossier"] = row.get("evidence_dossier", None)
    else:
        result["dossier"] = None
    
    return result

In [95]:
hidden_row = df[df["mismatch_type"] == "Hidden Crisis"].sample(1, random_state=10).iloc[0]

hidden_result = audit_existing_ticket(hidden_row)

hidden_result

{'ticket_id': 'TKT-101599',
 'model_prediction': 1,
 'model_prediction_label': 'Mismatch',
 'model_confidence': 0.9998538494110107,
 'prob_consistent': 0.00014610044308938086,
 'prob_mismatch': 0.9998538494110107,
 'dossier': {'ticket_id': 'TKT-101599',
  'assigned_priority': 'Medium',
  'inferred_severity': 'Critical',
  'mismatch_type': 'Hidden Crisis',
  'severity_delta': 2,
  'feature_evidence': [{'signal': 'keyword',
    'value': 'login failed',
    'weight': 3,
    'source_field': 'Ticket_Subject / Ticket_Description',
    'interpretation': 'Matched rule: login_failure'},
   {'signal': 'keyword',
    'value': 'data has not synced',
    'weight': 3,
    'source_field': 'Ticket_Subject / Ticket_Description',
    'interpretation': 'Matched rule: data_sync_issue'},
   {'signal': 'ticket_subject',
    'value': 'Login failed - Support',
    'source_field': 'Ticket_Subject',
    'interpretation': 'Ticket subject used as direct text evidence.'},
   {'signal': 'ticket_description',
    'v

In [96]:
false_row = df[df["mismatch_type"] == "False Alarm"].sample(1, random_state=10).iloc[0]

false_result = audit_existing_ticket(false_row)

false_result

{'ticket_id': 'TKT-113402',
 'model_prediction': 1,
 'model_prediction_label': 'Mismatch',
 'model_confidence': 0.9993589520454407,
 'prob_consistent': 0.0006410701898857951,
 'prob_mismatch': 0.9993589520454407,
 'dossier': {'ticket_id': 'TKT-113402',
  'assigned_priority': 'High',
  'inferred_severity': 'Low',
  'mismatch_type': 'False Alarm',
  'severity_delta': -2,
  'feature_evidence': [{'signal': 'keyword',
    'value': 'latest patch',
    'weight': 1,
    'source_field': 'Ticket_Subject / Ticket_Description',
    'interpretation': 'Matched rule: patch_information'},
   {'signal': 'ticket_subject',
    'value': 'Data not syncing - Expert',
    'source_field': 'Ticket_Subject',
    'interpretation': 'Ticket subject used as direct text evidence.'},
   {'signal': 'ticket_description',
    'value': 'Hi Support, How do I install the latest patch on Windows 11? Cold major available story floor here western west.',
    'source_field': 'Ticket_Description',
    'interpretation': 'Ticket 

In [97]:
consistent_row = df[df["mismatch_type"] == "Consistent"].sample(1, random_state=10).iloc[0]

consistent_result = audit_existing_ticket(consistent_row)

consistent_result

{'ticket_id': 'TKT-103061',
 'model_prediction': 0,
 'model_prediction_label': 'Consistent',
 'model_confidence': 0.9996978044509888,
 'prob_consistent': 0.9996978044509888,
 'prob_mismatch': 0.00030227116076275706,
 'dossier': None}

In [99]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("../data/processed/sia_dossiers.csv")

df_model = df.copy()

df_model["model_input"] = (
    "Assigned Priority: " + df_model["Priority_Level"].astype(str) +
    " | Issue Category: " + df_model["Issue_Category"].astype(str) +
    " | Ticket Channel: " + df_model["Ticket_Channel"].astype(str) +
    " | Resolution Time Hours: " + df_model["Resolution_Time_Hours"].astype(str) +
    " | Subject: " + df_model["Ticket_Subject"].astype(str) +
    " | Description: " + df_model["Ticket_Description"].astype(str)
)

df_model["label"] = df_model["mismatch_label"].astype(int)

train_df, temp_df = train_test_split(
    df_model,
    test_size=0.30,
    random_state=42,
    stratify=df_model["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

train_df.to_csv("../data/processed/train_split.csv", index=False)
val_df.to_csv("../data/processed/val_split.csv", index=False)
test_df.to_csv("../data/processed/test_split.csv", index=False)

print(train_df.shape, val_df.shape, test_df.shape)

(14000, 29) (3000, 29) (3000, 29)
